# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

In [10]:
import pandas as pd
# 1
# s = pd.Series([10, 20, 30], index=["a", "b", "c"])
# print(s["b"], s.iloc[0])
# 2
# a = pd.Series({"Алматы": 5, "Шымкент": 2})
# b = pd.Series({"Шымкент": 1, "Астана": 4})
# print(a + b)
# # 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# # 4
# print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# # 5
# f = df[df["x"] > 1]
# print(f.index.tolist(), f.iloc[0, 1])

   x  y
1  2  5
2  3  6


Ваши ответы и объяснения:

1.сначала по ключу 2 по индексу
2. склека Series там где не соотвествий nan
3. Создание DataFrame фильтр по икс больше 1 и вывести размерность
4. Выводит какие типы данных с 1 скобкой и с 2 скобками
5. Сначала фильтр потом вывод список а 2 функция вывод по индексу 0,1 значение 5

In [2]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [11]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

(5463, 8)


In [32]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
ALA = ["Алматы", "алматы", "almaty"]

print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

сентябрь: 2782 | октябрь: 2681


## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [24]:
# 1
df.info()
#Строк 5463
#Столбцов 8
print("Разница в delivery_days",str(df['order_id'].count()-df['delivery_days'].count()))
print("city ",df["city"].nunique())
print("category ",df["category"].nunique())
print("status ",df["status"].nunique())
print("customer_id ",df["customer_id"].nunique())
# Не берется в расчет что есть покупатель тест, отсуствуют данные по delivery_days, Некорректный справочник городов, есть дубли заказов

<class 'pandas.DataFrame'>
RangeIndex: 5463 entries, 0 to 5462
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       5463 non-null   int64  
 1   order_date     5463 non-null   str    
 2   customer_id    5463 non-null   str    
 3   city           5463 non-null   str    
 4   category       5463 non-null   str    
 5   amount         5463 non-null   float64
 6   delivery_days  5303 non-null   float64
 7   status         5463 non-null   str    
dtypes: float64(2), int64(1), str(5)
memory usage: 341.6 KB
Разница в delivery_days 160
city  14
category  5
status  2
customer_id  2003


## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [31]:
# 2
dfFilter=df[not_test]
print(dfFilter["amount"].mean())
print(dfFilter["amount"].median())
print(np.percentile(dfFilter["amount"], 25))
print(np.percentile(dfFilter["amount"], 75))
#Думаю средний чек показать честнее так как разброс сумм большой и что то среднее будет средняя сумма чем медиана

14630.657509157509
10320.0
5867.5
18680.0


## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [41]:
# 3
unpaid = ~paid
dfFilterCancel = df[unpaid & not_test]

rate_total = len(dfFilterCancel) / len(df[not_test])
rate_shymkent = len(df[unpaid & not_test & df["city"].isin(SH)]) / len(df[not_test & df["city"].isin(SH)])
rate_almaty = len(df[unpaid & not_test & df["city"].isin(ALA)]) / len(df[not_test & df["city"].isin(ALA)])

print(f"Общая доля отмен: {rate_total:.2%}")
print(f"Доля отмен в Шымкенте: {rate_shymkent:.2%}")
print(f"Доля отмен в Алматы: {rate_almaty:.2%}")

if rate_shymkent > rate_almaty:
    print(f"Выделяется Шымкент — доля отмен выше в {rate_shymkent / rate_almaty:.2f} раза.")
elif rate_almaty > rate_shymkent:
    print(f"Выделяется Алматы — доля отмен выше в {rate_almaty / rate_shymkent:.2f} раза.")

Общая доля отмен: 10.24%
Доля отмен в Шымкенте: 20.40%
Доля отмен в Алматы: 8.20%
Выделяется Шымкент — доля отмен выше в 2.49 раза.


## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [42]:
# 4
base = paid & not_test
shym_mask = df["city"].isin(SH)
ala_mask = df["city"].isin(ALA)

# Выручка: строки = [Шымкент, Алматы], столбцы = [Сентябрь, Октябрь]
rev = np.array([
    [df.loc[base & shym_mask & sep, "amount"].sum(), df.loc[base & shym_mask & octo, "amount"].sum()],
    [df.loc[base & ala_mask & sep, "amount"].sum(), df.loc[base & ala_mask & octo, "amount"].sum()],
])

# Изменение в долях: (Октябрь - Сентябрь) / Сентябрь
chg = (rev[:, 1] - rev[:, 0]) / rev[:, 0]

for name, (s_val, o_val), c_val in zip(["Шымкент", "Алматы"], rev, chg):
    print(f"{name:<10} | Сент: {s_val:>12,.0f} | Окт: {o_val:>12,.0f} | Изм: {c_val:>+7.1%}")

# Сравниваем падение (если оба города просели, берём величину падения со знаком плюс)
drop_shym = -chg[0] if chg[0] < 0 else 0.0
drop_ala = -chg[1] if chg[1] < 0 else 0.0

if drop_shym > 0 and drop_ala > 0:
    if drop_shym >= drop_ala:
        print(f"\nПадение сильнее в Шымкенте в {drop_shym / drop_ala:.2f} раза.")
    else:
        print(f"\nПадение сильнее в Алматы в {drop_ala / drop_shym:.2f} раза.")

Шымкент    | Сент:    4,949,800 | Окт:    4,044,010 | Изм:  -18.3%
Алматы     | Сент:   14,148,810 | Окт:   13,242,450 | Изм:   -6.4%

Падение сильнее в Шымкенте в 2.86 раза.


## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [43]:
# 5
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
#он пытется сначала выполнить "Шымкент" & df["status"]. Так как & имеет больший приоритет, чем ==

TypeError: unsupported operand type(s) for &: 'str' and 'StringArray'

## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [46]:
# 6
counts = df["customer_id"].value_counts()
counts

customer_id
C01818    11
C01076     9
C01947     9
C01656     9
C00420     9
          ..
C00645     1
C01442     1
C00018     1
C01650     1
C01057     1
Name: count, Length: 2003, dtype: int64

In [47]:
top_customer = counts.index[0]
print("Больше всего заказов у клиента:", top_customer, "| Всего строк:", counts.iloc[0])

Больше всего заказов у клиента: C01818 | Всего строк: 11


In [50]:
cols = ["order_id","order_date", "city", "amount", "status"]
top_orders = df.loc[df["customer_id"] == top_customer, cols]
top_orders

#Да есть повторяющиеся что приводит к недостоверным данным

,order_id,order_date,city,amount,status
57,101125,2026-09-01 15:00,Шымкент,7350.0,оплачен
779,100096,2026-09-09 18:00,Шымкент,45270.0,оплачен
1371,101504,2026-09-15 21:00,Алматы,8890.0,оплачен
1375,101504,2026-09-15 21:00,Алматы,8890.0,оплачен
2195,101860,2026-09-24 18:00,Астана,1490.0,оплачен
2617,101462,2026-09-29 09:00,Астана,22170.0,оплачен
2776,101495,2026-09-30 21:00,Алматы,20180.0,оплачен
3417,103416,2026-10-08 14:00,Караганда,4330.0,оплачен
4279,102644,2026-10-18 08:00,Астана,13600.0,оплачен
4610,104001,2026-10-21 19:00,Алматы,21990.0,оплачен


## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [ ]:
# 7

## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [ ]:
# 8